# Fine-tune RMBG-2.0 for Line Art Removal

Colab Pro GPU (100GB VRAM) optimized - Enhanced version

### 1. Clone project from GitHub

In [ ]:
!git clone https://github.com/hoangtung386/BRG-fine-tuning-model.git /content/fine_tune_model_BRG
%cd /content/fine_tune_model_BRG

In [ ]:
!pip install -qqq -r requirements.txt

### 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3. Check GPU & Install dependencies

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 4. Setup config for Colab

In [ ]:
import sys
sys.path.insert(0, '/content/fine_tune_model_BRG')

from config import config

# Cập nhật paths cho Colab
config.DATASET_PATH = '/content/drive/MyDrive/Projects/dataset_line-art'
config.MASK_PATH = '/content/drive/MyDrive/Projects/ground_truth_anime'
config.CKPT_DIR = "/content/drive/MyDrive/rmbg_checkpoints"

print("Config loaded:")
print(f"  IMG_SIZE: {config.IMG_SIZE}")
print(f"  BATCH_SIZE: {config.BATCH_SIZE}")
print(f"  NUM_EPOCHS: {config.NUM_EPOCHS}")
print(f"  LR (decoder): {config.LEARNING_RATE}")
print(f"  LR (encoder): {config.LEARNING_RATE_ENCODER}")
print(f"  USE_GRADIENT_CHECKPOINTING: {config.USE_GRADIENT_CHECKPOINTING}")
print(f"  CKPT_DIR: {config.CKPT_DIR}")

### 5. Load dataset (with augmentation)

In [ ]:
from torch.utils.data import DataLoader, random_split

from src.dataset import LineArtDataset

# Enable augmentation for training
dataset = LineArtDataset(config.DATASET_PATH, config.MASK_PATH, config.IMG_SIZE, augment=True)
train_len = int(config.TRAIN_RATIO * len(dataset))
val_len = len(dataset) - train_len
train_set, val_set = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(train_set, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS)
val_loader = DataLoader(val_set, batch_size=config.BATCH_SIZE, shuffle=False)

print(f"Train: {train_len} samples, Val: {val_len} samples")

### 6. Visualize sample data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Hiển thị 4 sample đầu tiên
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    img, mask = dataset[i]
    
    # Denormalize image
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    img_denorm = img * std + mean
    img_denorm = np.clip(img_denorm.transpose(1, 2, 0), 0, 1)
    
    axes[0, i].imshow(img_denorm)
    axes[0, i].set_title(f'Image {i+1}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(mask.squeeze(), cmap='gray')
    axes[1, i].set_title(f'Mask {i+1}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

### 7. Load model & optimizer (differential LR + gradient checkpointing)

In [ ]:
import torch
from transformers import AutoModelForImageSegmentation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model with gradient checkpointing
model = AutoModelForImageSegmentation.from_pretrained(config.MODEL_NAME, trust_remote_code=True).to(device)

if config.USE_GRADIENT_CHECKPOINTING and hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled")

for p in model.parameters():
    p.requires_grad = True

# Differential LR: encoder 5e-6, decoder 2e-5
encoder_params = []
decoder_params = []
for name, param in model.named_parameters():
    if 'encoder' in name.lower():
        encoder_params.append(param)
    else:
        decoder_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': encoder_params, 'lr': config.LEARNING_RATE_ENCODER},
    {'params': decoder_params, 'lr': config.LEARNING_RATE},
], weight_decay=config.WEIGHT_DECAY)

print(f"Model loaded: {config.MODEL_NAME}")
print(f"Device: {device}")
print(f"Encoder params: {len(encoder_params)}, LR: {config.LEARNING_RATE_ENCODER}")
print(f"Decoder params: {len(decoder_params)}, LR: {config.LEARNING_RATE}")

### 8. Test prediction before training

In [ ]:
from src.visualization import visualize_prediction

# Test predict 1 sample trước khi train
visualize_prediction(model, dataset, idx=0, device=device)

### 9. Start Training (with SSIM + Boundary IoU)

In [ ]:
from src.trainer import Trainer

print("Starting training...")
print("Loss: 10×SSIM + 90×BCE + 0.25×IoU")
print("Best model saved by Boundary IoU (5px edge)\n")

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    device=device,
    ckpt_dir=config.CKPT_DIR,
    num_epochs=config.NUM_EPOCHS,
    use_boundary_iou=True
)
trainer.train(train_loader, val_loader)

### 10. Visualize predictions after training

In [ ]:
# Hiển thị kết quả predict sau khi train
visualize_prediction(model, dataset, idx=0, device=device)
visualize_prediction(model, dataset, idx=5, device=device)

### 11. Visualize batch predictions

In [ ]:
from src.visualization import visualize_batch

# Hiển thị batch predictions
visualize_batch(model, val_loader, device=device, n_images=4)

### 12. Compute final metrics

In [ ]:
from src.visualization import compute_metrics_batch
from src.losses import boundary_iou

model.eval()
total_iou = 0
total_boundary_iou = 0

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(device)
        masks = masks.to(device)
        preds = model(imgs)[-1]
        total_iou += compute_metrics_batch(model, val_loader, device=device)
        # Note: compute_metrics_batch already returns IoU

val_iou = total_iou / len(val_loader)
print(f"Final Validation IoU: {val_iou:.4f}")
print(f"Final Best Boundary IoU: {trainer.best_boundary_iou:.4f}")

### 13. Verify checkpoints saved to Drive

In [ ]:
import os

print("Checkpoints in Drive:")
for f in os.listdir(config.CKPT_DIR):
    size = os.path.getsize(os.path.join(config.CKPT_DIR, f)) / 1e6
    print(f"  {f}: {size:.2f} MB")